In [1]:
import numpy as np 
import pandas as pd 
import math
import random
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split, cross_val_predict
from sklearn import metrics
from sklearn import svm
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import roc_curve, auc
from sklearn.metrics import roc_auc_score
from sklearn.tree import DecisionTreeClassifier
from sklearn import tree
from sklearn.dummy import DummyClassifier
from imblearn.metrics import geometric_mean_score
from imblearn.over_sampling import SMOTE
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from collections import Counter

In [2]:
data=pd.read_csv("UNSW_NB15_training-set.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'UNSW_NB15_training-set.csv'

In [ ]:
import pandas

In [ ]:
data=pd.read_csv("UNSW_NB15_training-set.csv")

In [ ]:
data

In [ ]:
data = data.drop(columns=['service', 'label'])

In [ ]:
data

In [ ]:
data.proto = pd.factorize(data.proto)[0] + 1

data.state = pd.factorize(data.state)[0] + 1

In [ ]:
gb=data.groupby("attack_cat")

In [ ]:
gb.size()

In [ ]:
data.info()

In [ ]:
data.attack_cat = pd.factorize(data.attack_cat)[0] + 1

In [ ]:
y = data['attack_cat']

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(data, y, train_size=.7)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

feature_names = [f"feature {i}" for i in range(data.shape[1])]
forest = RandomForestClassifier(random_state=0)
forest.fit(X_train, Y_train)
importances = forest.feature_importances_
std = np.std([tree.feature_importances_ for tree in forest.estimators_], axis=0)

In [ ]:
forest_importances = pd.Series(importances, index=feature_names)

#fig, ax = plt.subplots()
fig, ax = plt.subplots(figsize=(17, 6))
forest_importances.plot.bar(yerr=std, ax=ax)
ax.set_title("Feature importances using MDI")
ax.set_ylabel("Mean decrease in impurity")
fig.tight_layout()
#plt.show()

In [ ]:
new_data = data.drop(['stcpb', 'dtcpb', 'dwin', 'trans_depth', 'response_body_len', 'is_ftp_login', 'ct_ftp_cmd', 'ct_flw_http_mthd', 'is_sm_ips_ports'], axis=1, inplace=True)

In [ ]:
data.info()

In [ ]:
y = data['attack_cat']

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(data, y, train_size=.7)

In [ ]:
feature_names = [f"feature {i}" for i in range(data.shape[1])]
forest = RandomForestClassifier(random_state=0)
forest.fit(X_train, Y_train)
importances = forest.feature_importances_
std = np.std([tree.feature_importances_ for tree in forest.estimators_], axis=0)

In [ ]:
forest_importances = pd.Series(importances, index=feature_names)

#fig, ax = plt.subplots()
fig, ax = plt.subplots(figsize=(17, 6))
forest_importances.plot.bar(yerr=std, ax=ax)
ax.set_title("Feature importances using MDI")
ax.set_ylabel("Mean decrease in impurity")
fig.tight_layout()
#plt.show()

In [ ]:
gb=data.groupby("attack_cat")

In [ ]:
gb.size()

In [ ]:
gb.get_group(1)

In [ ]:
result = gb.get_group(1).append([gb.get_group(4), gb.get_group(5), gb.get_group(7), gb.get_group(10) ], ignore_index='True')

In [ ]:
#Spliting the dataset into groups according to the label attribute
new_gb = result.groupby("attack_cat")

In [ ]:
new_gb.size()

In [ ]:
target = result['attack_cat']

In [ ]:
result

In [ ]:
target

In [ ]:
#undersampling
rus = RandomUnderSampler(random_state=42)

X_res, y_res = rus.fit_resample(result, target)

In [ ]:
print('Resampled dataset shape %s' % Counter(y_res))

In [ ]:
X_res

In [ ]:
gb_new_SMOTE = X_res.groupby("attack_cat")

In [ ]:
gb_new_SMOTE.size()

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(X_res, y_res, train_size=.7)

In [ ]:
#Loading the Classifier model
model=svm.SVC(C = 1, probability=True)
clf = DecisionTreeClassifier(random_state=0, max_depth=2)
knn = KNeighborsClassifier(n_neighbors=2)

In [ ]:
#Train the model using the training sets
model.fit(X_train, Y_train)
clf = clf.fit(X_train, Y_train)
knn = knn.fit(X_train, Y_train)

In [ ]:
#prule_pred_proba=prule.predict_proba(X_test)
model_pred_proba=model.predict_proba(X_test)
clf_pred_proba=clf.predict_proba(X_test)
knn_pred_proba=knn.predict_proba(X_test)

In [ ]:
model_X_train_pred=model.predict(X_train)
model_X_test_pred=model.predict(X_test)
clf_X_train_pred=clf.predict(X_train)
clf_X_test_pred=clf.predict(X_test)
knn_X_train_pred=knn.predict(X_train)
knn_X_test_pred=knn.predict(X_test)

In [ ]:
print("Accuracy for SVM Test Data:                        ", metrics.accuracy_score(Y_test, model_X_test_pred))
print("Accuracy for Decision Tree Classifier Test Data:   ", metrics.accuracy_score(Y_test, clf_X_test_pred))
print("Accuracy for KNN Test Data:                        ", metrics.accuracy_score(Y_test, knn_X_test_pred))

In [ ]:
model_gmean=geometric_mean_score(Y_test, model_X_test_pred, average='weighted')
clf_gmean=geometric_mean_score(Y_test, clf_X_test_pred, average='weighted')
knn_gmean=geometric_mean_score(Y_test, knn_X_test_pred, average='weighted')

In [ ]:
print("G mean for SVM Test Data:                           ", model_gmean)
print("G mean for Decision Tree Classifier Test Data:      ", clf_gmean)
print("G mean for KNN Test Data:                           ", knn_gmean)

In [ ]:
### Decision Tree

In [ ]:
cm_model_test=confusion_matrix(Y_test, clf_X_test_pred)

In [ ]:
cm_model_test

In [ ]:
disp_model = ConfusionMatrixDisplay(confusion_matrix=cm_model_test, display_labels=model.classes_)

In [ ]:
disp_model.plot() 

In [ ]:
print(metrics.classification_report(Y_test, clf_X_test_pred, digits=3))

In [ ]:
### KNN

In [ ]:
knn_model_test=confusion_matrix(Y_test, knn_X_test_pred)

In [ ]:
knn_model_test

In [ ]:
disp_model = ConfusionMatrixDisplay(confusion_matrix=knn_model_test, display_labels=model.classes_)

In [ ]:
disp_model.plot()

In [ ]:
print(metrics.classification_report(Y_test, knn_X_test_pred, digits=3))

In [ ]:
### SVM

In [ ]:
model_X_test=confusion_matrix(Y_test, model_X_test_pred)

In [ ]:
model_X_test

In [ ]:
disp_model = ConfusionMatrixDisplay(confusion_matrix=model_X_test, display_labels=model.classes_)

In [ ]:
disp_model.plot()

In [ ]:
print(metrics.classification_report(Y_test, model_X_test_pred, digits=3))